In [ ]:
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as font_manager
import urllib.request
from statsmodels.nonparametric.smoothers_lowess import lowess

if not os.path.exists('IBMPlexMono-Regular.ttf'):
    urllib.request.urlretrieve(
        'https://github.com/google/fonts/raw/main/ofl/ibmplexmono/IBMPlexMono-Regular.ttf',
        'IBMPlexMono-Regular.ttf'
    )
fe = font_manager.FontEntry(fname='IBMPlexMono-Regular.ttf', name='plexmono')
font_manager.fontManager.ttflist.append(fe)

plt.rcParams.update({
    'axes.facecolor': '#f5f4e9',
    'grid.color':     '#AAAAAA',
    'axes.edgecolor': '#333333',
    'figure.facecolor': '#FFFFFF',
    'axes.grid': True,
    'axes.prop_cycle': plt.cycler('color', plt.cm.Dark2.colors),
    'font.family': fe.name,
    'figure.dpi':  150,
    'ytick.left':  True,
    'xtick.bottom': True,
    'xtick.labelsize': 14, # modified
    'ytick.labelsize': 14, # modified
})

os.makedirs('paper_figs/tradeoff_analysis', exist_ok=True)

In [ ]:
censor_region = 'above'
censor_splits = [0.1, 0.5, 0.9]
run_date      = '2026-03-14'
censor_types  = ['omit', 'xnoise', 'ynoise']
#ctype_labels  = ['Omission', r'Feature Noise ($\delta X$)', r'Label Noise ($\delta y$)']
ctype_labels  = ['Omission', 'Feature Noise', 'Label Noise']
ctype_markers = ['o', 's', '^']
NT_COLORS     = {'omit': 'C6', 'xnoise': 'C2', 'ynoise': 'C3'}
colors        = [NT_COLORS[ctype] for ctype in censor_types]
results_dir   = '../all_results'
LOWESS_FRAC   = 0.4

In [ ]:
def calculate_x_noise_levels(intervals):
    levels = []
    for c in intervals:
        s1, s2 = map(float, c.split('-'))
        levels.append(round(1 - (s1 + s2) / 2, 3))
    return levels

def load_data(ctype, split):
    path = f'{results_dir}/gcn_{ctype}_results_split{split}_{censor_region}/dataframe_{run_date}.json'
    df   = pd.read_json(path)
    noise_levels = df.iloc[:, 0].tolist()
    if ctype == 'xnoise':
        noise_levels = calculate_x_noise_levels(noise_levels)
    return {
        'noise_levels':   np.array(noise_levels, dtype=float),
        'upper_corr':     np.array(df['upper corr']),
        'lower_corr':     np.array(df['lower corr']),
        'upper_corr_std': np.array(df['upper corr std']),
        'lower_corr_std': np.array(df['lower corr std']),
    }

all_data = {ctype: {split: load_data(ctype, split)
                    for split in censor_splits}
            for ctype in censor_types}

In [ ]:
def normalize_noise(noise):
    mn, mx = noise.min(), noise.max()
    return (noise - mn) / (mx - mn) if mx > mn else np.zeros_like(noise)

def pareto_indices(utility, security):
    """Indices of non-dominated points (maximize both utility and security)."""
    pts = sorted(zip(utility, security, range(len(utility))), key=lambda x: -x[0])
    best_sec, pareto = -np.inf, []
    for u, s, idx in pts:
        if s > best_sec:
            pareto.append(idx)
            best_sec = s
    return sorted(pareto)

def format_noise(ctype, val):
    if ctype == 'omit':   return f'{val:.0%}'
    if ctype == 'xnoise': return f'{val:.3f}' if val > 0.6 else f'{val:.2f}'
    return f'{val:.1f}'

PREFIX = {'omit': 'O', 'xnoise': 'X', 'ynoise': 'Y'}

def tradeoff_axes_labels(axs):
    for i, (ax, split) in enumerate(zip(axs, censor_splits)):
        ax.set_title(f'{split*100:.0f}% sensitive split', fontsize=14)
        if i == 1:
            ax.set_xlabel('Non-sensitive Corr.\n(utility → higher is better)', fontsize=16)
        ax.grid(True, alpha=0.4)
    axs[0].set_ylabel('Sensitive Corr. Degradation\n(security ↑ higher is better)', fontsize=16)

# Per-point annotation offsets to fix overlaps
# Key: (split, label)  e.g. (0.9, 'Y:1.0')  Value: (x, y) in points.
ANNOTATION_OFFSETS = {
    (0.1, 'X:0.25'):  (-50, -8),
    (0.1, 'O:100%'):  (4, 4),
    (0.1, 'Y:1.2'): (6, -8),
    (0.1, 'Y:0.4'): (-16, -16),

    (0.5, 'O:100%'):  (4, 4),
    (0.5, 'O:60%'):  (4, -16),
    (0.5, 'X:0.675'): (4, -16),

    (0.9, 'X:0.725'): (4, -16),
}
DEFAULT_OFFSET = (4, 4) # default is x=4, y=4 from the marker (up and right)


---
## Main figure — Pareto frontier with LOWESS background

Background: LOWESS-smoothed curves (raw points at α = 0.2).  
Frontier computed on actual data points. Each frontier marker uses the method's shape  
and is annotated with a prefix (O/X/Y) and noise level.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(14.5, 4.5))

for i, split in enumerate(censor_splits):
    ax = axs[i]
    all_u, all_s, all_j, all_n = [], [], [], []

    for j, ctype in enumerate(censor_types):
        d          = all_data[ctype][split]
        utility    = d['lower_corr']
        security   = d['upper_corr'][0] - d['upper_corr']
        noise      = d['noise_levels']
        noise_norm = normalize_noise(noise)

        util_s = lowess(utility,  noise_norm, frac=LOWESS_FRAC, return_sorted=False)
        sec_s  = lowess(security, noise_norm, frac=LOWESS_FRAC, return_sorted=False)

        ax.plot(utility, security, color=colors[j], lw=0.6, alpha=0.2,
                marker=ctype_markers[j], ms=2)
        ax.plot(util_s, sec_s, color=colors[j], lw=1.8, alpha=0.6,
                label=ctype_labels[j])

        all_u.extend(utility.tolist()); all_s.extend(security.tolist())
        all_j.extend([j] * len(utility)); all_n.extend(noise.tolist())

    all_u = np.array(all_u); all_s = np.array(all_s)
    all_j = np.array(all_j); all_n = np.array(all_n)

    pidx_sorted = np.array(pareto_indices(all_u, all_s))
    pidx_sorted = pidx_sorted[np.argsort(all_u[pidx_sorted])]
    ax.plot(all_u[pidx_sorted], all_s[pidx_sorted],
            color='black', lw=1.5, ls='--', zorder=4)
    if i == 0:
        ax.set_xlim(0.65, 0.75)
    if i == 1:
        ax.set_xlim(0, 0.7)
    if i == 2:
        ax.set_xlim(0.15, 0.5)

    for k in pidx_sorted:
        j    = all_j[k]
        u, s = all_u[k], all_s[k]
        ax.scatter(u, s, color=colors[j], marker=ctype_markers[j], s=90,
                   zorder=5, edgecolors='black', linewidths=0.7)
        lbl = f"{PREFIX[censor_types[j]]}:{format_noise(censor_types[j], all_n[k])}"
        ax.annotate(lbl, (u, s), textcoords='offset points', xytext=ANNOTATION_OFFSETS.get((split, lbl), DEFAULT_OFFSET),
                    fontsize=11, color='black', fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.1', facecolor='white',
                              alpha=0.7, edgecolor='none'))

tradeoff_axes_labels(axs)

legend_handles = [
    plt.Line2D([0],[0], marker=ctype_markers[j], color=colors[j], lw=1.5, ms=5,
               label=ctype_labels[j])
    for j in range(len(censor_types))
] + [plt.Line2D([0],[0], color='black', lw=1.5, ls='--', ms=6,
                label='Pareto frontier')]
fig.legend(handles=legend_handles, loc='center right', bbox_to_anchor=(0.97, 0.02), fontsize=12)
plt.tight_layout()
plt.savefig('paper_figs/tradeoff_pareto.pdf', bbox_inches='tight')
plt.show()

---
## SI — Pareto frontier on raw (unsmoothed) data

Same frontier as the main figure, plotted over unsmoothed experimental data.  
Shows that frontier points are anchored to actual measurements.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(14, 4.5))

for i, split in enumerate(censor_splits):
    ax = axs[i]
    all_u, all_s, all_j, all_n = [], [], [], []

    for j, ctype in enumerate(censor_types):
        d        = all_data[ctype][split]
        utility  = d['lower_corr']
        security = d['upper_corr'][0] - d['upper_corr']
        noise    = d['noise_levels']

        ax.plot(utility, security, color=colors[j], lw=1.2, alpha=0.5,
                marker=ctype_markers[j], ms=3, label=ctype_labels[j])

        all_u.extend(utility.tolist()); all_s.extend(security.tolist())
        all_j.extend([j] * len(utility)); all_n.extend(noise.tolist())
        
    if i == 0:
        ax.set_xlim(0.65, 0.75)
    if i == 1:
        ax.set_xlim(0, 0.7)
    if i == 2:
        ax.set_xlim(0.15, 0.5)

    all_u = np.array(all_u); all_s = np.array(all_s)
    all_j = np.array(all_j); all_n = np.array(all_n)

    pidx_sorted = np.array(pareto_indices(all_u, all_s))
    pidx_sorted = pidx_sorted[np.argsort(all_u[pidx_sorted])]
    ax.plot(all_u[pidx_sorted], all_s[pidx_sorted],
            color='black', lw=1.5, ls='--', zorder=4)

    # for k in pidx_sorted:
    #     j    = all_j[k]
    #     u, s = all_u[k], all_s[k]
    #     ax.scatter(u, s, color=colors[j], marker=ctype_markers[j], s=90,
    #                zorder=5, edgecolors='black', linewidths=0.7)
    #     lbl = f"{PREFIX[censor_types[j]]}:{format_noise(censor_types[j], all_n[k])}"
    #     ax.annotate(lbl, (u, s), textcoords='offset points', xytext=ANNOTATION_OFFSETS.get((split, lbl), DEFAULT_OFFSET),
    #                 fontsize=11, color='black', fontweight='bold',
    #                 bbox=dict(boxstyle='round,pad=0.1', facecolor='white',
    #                           alpha=0.7, edgecolor='none'))

tradeoff_axes_labels(axs)

fig.legend(handles=legend_handles, loc='center right', bbox_to_anchor=(0.97, 0.02), fontsize=12)
plt.tight_layout()
plt.show()

---
## SI — Normalised % change from baseline

Security gain (% drop in sensitive corr) and utility cost (% drop in non-sensitive corr)  
as a function of noise level. Green shading: security gain > utility cost.

In [ ]:
fig, axs = plt.subplots(3, 3, figsize=(14, 11))

for row, ctype in enumerate(censor_types):
    for col, split in enumerate(censor_splits):
        ax   = axs[row, col]
        d    = all_data[ctype][split]
        noise = d['noise_levels']
        bu, bl = d['upper_corr'][0], d['lower_corr'][0]

        sec = (bu - d['upper_corr']) / abs(bu) * 100
        utl = (bl - d['lower_corr']) / abs(bl) * 100

        ax.plot(noise, sec, color='C1', marker='x', lw=1.5, ms=4,
                label='Security gain (sensitive ↓%)')
        ax.plot(noise, utl, color='C0', marker='^', lw=1.5, ms=4,
                label='Utility cost (non-sensitive ↓%)')
        ax.fill_between(noise, sec, utl, where=sec >  utl, alpha=0.15, color='green')
        ax.fill_between(noise, sec, utl, where=sec <= utl, alpha=0.15, color='red')
        ax.axhline(0, color='black', lw=0.5, ls='--')

        if row == 0: ax.set_title(f'{split*100:.0f}% sensitive split', fontsize=12)
        if col == 0: ax.set_ylabel(f'{ctype_labels[row]}\n% Change from Baseline', fontsize=12)
        if row == 2: ax.set_xlabel('Noise level', fontsize=16)
        ax.grid(True, alpha=0.4)

leg_handles = [
    plt.Line2D([0],[0], color='C1', marker='x', lw=1.5, ms=4, label='Security gain (sensitive ↓%)'),
    plt.Line2D([0],[0], color='C0', marker='^', lw=1.5, ms=4, label='Utility cost (non-sensitive ↓%)'),
    plt.Rectangle((0,0),1,1, color='green', alpha=0.3, label='Security > Utility'),
    plt.Rectangle((0,0),1,1, color='red',   alpha=0.3, label='Utility ≥ Security'),
]
fig.legend(handles=leg_handles, loc='lower center',
           bbox_to_anchor=(0.5, -0.04), ncol=4, fontsize=9)
plt.tight_layout()
plt.show()